#Identify outliers while being blinded to condition

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import seaborn as sns
import sys
src_path = str(Path.cwd().parent)
if src_path not in sys.path:
    sys.path.append(src_path)
from microscopy_analysis.d01_init_proc.vis_and_rescale import show_figs
from microscopy_analysis.d00_utils import utilities as utils

In [ ]:
df_path = Path(input())

In [ ]:
df = pd.read_csv(df_path)
df.head()

In [ ]:
outliers_csvname = f'outliers.csv'
outliers_df_path = df_path.parent / outliers_csvname

In [ ]:
def get_outliers(df, colname, splitby, iqr_factor=1.5):
    
    conds = np.unique(df[splitby])
    
    df[f'{colname}_outlier'] = False

    for cond in conds:

        col_cond = df.loc[(df[splitby]==cond), colname]

        # Calculate quantiles
        q1 = col_cond.quantile(0.25)
        q3 = col_cond.quantile(0.75)
        iqr = q3 - q1
    
        # Calculate upper and lower bounds
        upper_bound = q3 + (iqr_factor * iqr)
        lower_bound = q1 - (iqr_factor * iqr)
        
        print(f'{colname} upper bound ({cond}): {upper_bound}')
        print(f'lower bound ({cond}): {lower_bound}')
        
        df.loc[((df[splitby]==cond) & (df[colname] > upper_bound)), f'{colname}_outlier'] = True
        df.loc[((df[splitby]==cond) & (df[colname] < lower_bound)), f'{colname}_outlier'] = True
    
    return df

In [ ]:
iqr_factor=1

df = get_outliers(df, 'mean actin int (caax)', splitby='tx', iqr_factor=iqr_factor)
df = get_outliers(df, 'mean actin int (cell)', splitby='tx', iqr_factor=iqr_factor)
df = get_outliers(df, 'mean actin int (compacted)', splitby='tx', iqr_factor=iqr_factor)
df = get_outliers(df, 'mean actin int (caax)', splitby='tx', iqr_factor=iqr_factor)

In [ ]:
df.head()

In [ ]:
outlier_cols = [col for col in df.columns if 'outlier' in col]
outlier_cols

In [ ]:
df['outlier (any)'] = False
for col in outlier_cols:
    df.loc[df[col]==True, 'outlier (any)'] = True
outliers_df = df[df['outlier (any)']==True]

In [ ]:
# blinding: drop all columns showing info
cols_to_drop = ['DIV', 'tx']
outliers_df = outliers_df.drop(cols_to_drop, axis=1)

In [ ]:
cols_to_add = ['omit', 'omit reason', 'actin omit', 'actin omit reason']
for col in cols_to_add:
    outliers_df[col] = ''

In [ ]:
utils.safe_save_csv(outliers_df, outliers_df_path)

In [ ]:
outliers_df = pd.read_csv(outliers_df_path)
outliers_df

In [ ]:
# get unique ID of all cells marked to be omitted for actin analysis
outliers_UIDs = outliers_df.loc[outliers_df['actin omit']=='Y', ['UID', 'actin omit', 'actin omit reason']]
outliers_UIDs

In [ ]:
df = df.merge(outliers_UIDs, how='left', on='UID')
df

In [ ]:
utils.safe_save_csv(df, df_path)